# Resultados locales (escenarios livianos)

Tarea 2 — CSP, Metaheurísticas y Búsqueda con Adversarios.

Este notebook corre y muestra **solo los escenarios baratos** de la tarea, directamente sobre `src/problems/{nqueens,graph_coloring}.py` (sin pasar por `experiments/run_*.py`, que mezclan tamaños chicos y grandes en un mismo script):

- N-reinas con N=8 (backtracking y recocido simulado).
- Coloreado de grafos con 50 nodos (backtracking, recocido simulado y OR-Tools).

**Los escenarios pesados — N-reinas N=100 (incluida la enumeración de todas las soluciones) y coloreado de 1000 nodos — se corrieron en el servidor remoto** (`diego@132.248.52.48`), no aquí, para no dejar la laptop ocupada con búsquedas que pueden tardar minutos por corrida. Ese servidor clonó este mismo repo y corrió `scripts/run_remote_heavy.sh` con `nohup` en background; sus resultados (`results/tables/*.csv`, `results/solutions/*`) se trajeron de vuelta con `rsync` y quedan en `results/` igual que si se hubieran generado aquí — el reporte los usa sin distinguir de dónde vinieron.

In [ ]:
import sys
from pathlib import Path

# Notebook vive en notebooks/, pero los imports "from src..." esperan correr
# desde la raíz del repo — igual que _common.py hace para experiments/*.py.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.integrations.ortools_coloring import find_min_colors_ortools
from src.problems import graph_coloring, nqueens
from src.utils.graph_io import (
    generate_random_graph,
    write_coloring,
    write_graph,
)
from src.utils.metrics import append_result_csv, save_solution_json
from src.utils.validation import is_valid_coloring, is_valid_nqueens_solution

RESULTS = REPO_ROOT / "results"
SEED = 42

## N-reinas, N=8

In [ ]:
positions_bt, stats_bt = nqueens.solve_backtracking(
    n=8, use_forward_checking=True, use_ac3=False, time_limit_seconds=30
)
assert is_valid_nqueens_solution(positions_bt)
print("backtracking:", positions_bt)
print(f"  solved={stats_bt.solved} time={stats_bt.time_seconds:.4f}s")

positions_sa, stats_sa = nqueens.solve_metaheuristic(
    n=8,
    seed=SEED,
    initial_temperature=10.0,
    cooling_rate=0.995,
    max_iterations=100_000,
    time_limit_seconds=30,
)
print("\nrecocido simulado:", positions_sa)
print(
    f"  conflictos={nqueens.count_conflicts(positions_sa)} time={stats_sa.time_seconds:.4f}s"
)

append_result_csv(RESULTS / "tables" / "nqueens.csv", stats_bt)
append_result_csv(RESULTS / "tables" / "nqueens.csv", stats_sa)
save_solution_json(
    RESULTS / "solutions" / "nqueens_8_backtracking.json",
    positions_bt,
    stats_bt,
)
save_solution_json(
    RESULTS / "solutions" / "nqueens_8_metaheuristic.json",
    positions_sa,
    stats_sa,
)

## Coloreado de grafos, 50 nodos

Mismo grafo aleatorio (semilla fija) que usa `experiments/run_graph_coloring.py` para la instancia "small" (`edge_prob=0.1`). Se busca el menor `k` factible en `[2, 12]` con cada método.

In [ ]:
GRAPH_PATH = REPO_ROOT / "data" / "graphs" / "graph_50.txt"
if not GRAPH_PATH.exists():
    num_vertices, edges = generate_random_graph(50, edge_prob=0.1, seed=SEED)
    write_graph(GRAPH_PATH, num_vertices, edges)
else:
    from src.utils.graph_io import read_graph

    num_vertices, edges = read_graph(GRAPH_PATH)

print(f"{num_vertices} vertices, {len(edges)} edges")

In [ ]:
K_MIN, K_MAX = 2, 12

coloring_bt = stats_bt = k_bt = None
for k in range(K_MIN, K_MAX + 1):
    coloring_bt, stats_bt = graph_coloring.solve_backtracking(
        num_vertices,
        edges,
        k,
        use_forward_checking=True,
        use_ac3=False,
        time_limit_seconds=60,
    )
    if coloring_bt is not None:
        k_bt = k
        break

assert coloring_bt is not None and is_valid_coloring(edges, coloring_bt)
print(f"backtracking: k_min={k_bt} time={stats_bt.time_seconds:.4f}s")

coloring_sa = stats_sa = k_sa = None
for k in range(K_MIN, K_MAX + 1):
    coloring_sa, stats_sa = graph_coloring.solve_metaheuristic(
        num_vertices,
        edges,
        k,
        seed=SEED,
        initial_temperature=10.0,
        cooling_rate=0.995,
        max_iterations=100_000,
        time_limit_seconds=60,
    )
    if graph_coloring.count_conflicts(edges, coloring_sa) == 0:
        k_sa = k
        break

print(f"metaheuristica: k_min={k_sa} time={stats_sa.time_seconds:.4f}s")

coloring_or, stats_or = find_min_colors_ortools(
    num_vertices, edges, K_MIN, K_MAX, time_limit_seconds=60
)
print(f"OR-Tools: k={stats_or.get('k')} time={stats_or.get('time_seconds'):.4f}s")

append_result_csv(RESULTS / "tables" / "graph_coloring.csv", stats_bt)
append_result_csv(RESULTS / "tables" / "graph_coloring.csv", stats_sa)
write_coloring(RESULTS / "solutions" / "coloring_small_backtracking.txt", coloring_bt)
write_coloring(RESULTS / "solutions" / "coloring_small_metaheuristic.txt", coloring_sa)

## Resumen

| Escenario | Método | k / conflictos | Tiempo |
|---|---|---|---|
| N=8 reinas | Backtracking+FC | 0 conflictos | ver arriba |
| N=8 reinas | Recocido simulado | ver arriba | ver arriba |
| Coloreado 50 nodos | Backtracking+FC | `k_bt` | ver arriba |
| Coloreado 50 nodos | Recocido simulado | `k_sa` | ver arriba |
| Coloreado 50 nodos | OR-Tools | `stats_or["k"]` | ver arriba |

Para N=100 reinas (backtracking + metaheurística + enumeración exhaustiva acotada) y coloreado de 1000 nodos (backtracking + metaheurística + OR-Tools), ver `results/tables/nqueens.csv`, `results/tables/nqueens_enumeration.csv`, `results/tables/graph_coloring.csv` y `results/tables/coloring_comparison.csv` — generados en el servidor remoto vía `scripts/run_remote_heavy.sh` y traídos con `rsync` (ver README).